# Model Training Evaluation
This notebook visualizes the training history of the PointNet++ segmentation model using the exported CSV logs.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Use a professional plotting style
try:
    sns.set_theme(style="whitegrid", palette="muted")
except:
    plt.style.use('ggplot')

%matplotlib inline

## 1. Load Training History
We point to the CSV file generated during the training run.

In [ ]:
csv_path = 'checkpoints/pointnet2seg_segmentation_history.csv'

if not os.path.exists(csv_path):
    print(f"Error: {csv_path} not found. Please ensure training has completed or been logged.")
else:
    df = pd.read_csv(csv_path)
    display(df.head())
    display(df.describe())

## 2. Loss Curves
Visualizing training vs validation loss to check for convergence or overfitting.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df['epoch'], df['train_loss'], label='Train Loss', linewidth=2, marker='o', markersize=4)
plt.plot(df['epoch'], df['val_loss'], label='Val Loss', linewidth=2, marker='s', markersize=4)

plt.title('Training and Validation Loss', fontsize=16, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('results/loss_curves.png', dpi=300)
plt.show()

## 3. Intersection over Union (IoU) Metrics
Comparing the overall mIoU and the specific Anomaly IoU (the class we care about most).

In [ ]:
plt.figure(figsize=(10, 6))

# Plot mIoU (val_score) and Anomaly IoU
plt.plot(df['epoch'], df['val_score'], label='Mean IoU (mIoU)', color='forestgreen', linewidth=2)
plt.plot(df['epoch'], df['val_anomaly_iou'], label='Anomaly IoU', color='crimson', linewidth=2, linestyle='--')

plt.title('Segmentation Quality Metrics', fontsize=16, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('IoU Score', fontsize=12)
plt.ylim(0, 1.0)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('results/iou_performance.png', dpi=300)
plt.show()

## 4. Summary Table
Quickly find the best epoch results for your report.

In [ ]:
best_miou = df.loc[df['val_score'].idxmax()]
best_anomaly = df.loc[df['val_anomaly_iou'].idxmax()]

print("BEST PERFORMANCE SUMMARY")
print("-" * 30)
print(f"Best mIoU:        {best_miou['val_score']:.4f} at Epoch {int(best_miou['epoch'])}")
print(f"Best Anomaly IoU: {best_anomaly['val_anomaly_iou']:.4f} at Epoch {int(best_anomaly['epoch'])}")
print(f"Final Val Loss:   {df.iloc[-1]['val_loss']:.4f}")